# 06e — Ordering vs frequency: decisive tests (G=28)

Track A of `RESULTS_HANDOFF_barycenters.md` §12.6. Builds on **06c** (loads its cached G=28
artifacts). Two decisive, low-degree-of-freedom tests that separate *"does temporal ordering
carry group signal beyond symbol frequency?"* from the barycenter machinery:

1. **Incremental-AUC of ordering over frequency** — a plain classifier (logistic + RF, nested CV)
   on `[unigram histogram]` vs `[histogram ⊕ bigram transitions]`; report the *incremental* held-out
   AUC of ordering, per comparison. No barycenter confound.
2. **Pre-registered TBI-vs-RIL follow-up** — Edit-Shape DTW (+ bigram) at full (segment-level)
   length, `step_sequ=1`, 10 splits, **bootstrap 95% CIs**.

### Pre-registration (stated before any result)
> **Single hypothesis (TBI-vs-RIL only):** at segment-level / `step_sequ=1`, the **Edit-Shape DTW**
> AUC on `RIL_vs_TBI` beats the Wasserstein **histogram**, with a *paired bootstrap 95% CI on the
> delta (eshape − histogram) whose lower bound > 0.*
>
> Decision rule (no p-hacking): the follow-up is "positive" **iff** `delta_ci_low > 0`. The bigram
> transition method is reported as a secondary, exploratory ordering signal. For Goal 1, "ordering
> helps" only if the incremental-Δ CI excludes 0. Negatives are reported plainly and hand off to
> Track B (paper reframing).

In [ ]:
%load_ext autoreload
%autoreload 2
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
import smartflat
from joblib import load
from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.constants import incomplete_clinical_administrations
from smartflat.features.symbolization.utils import fix_clinical_diagnosis
from smartflat.features.symbolic_barycenter import vocab
from smartflat.features.symbolic_barycenter import baselines as B
from smartflat.features.symbolic_barycenter.baselines import make_patient_control_labels
plt.rcParams['figure.dpi'] = 110

ANN, RND, CFG = 'samperochon', 8, 'SymbolicSourceInferenceGoldConfig'
L, NU, LMBDA, OFFSET = 64, 1e-4, 0.1, 0.3
OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'g28')
EXP = os.path.join(OUT, 'experiments')
os.makedirs(EXP, exist_ok=True)


In [ ]:
# --- load cached G=28 artifacts from 06c and rebuild the analysis cohort ---
dff = load(os.path.join(OUT, 'g28_full_with_cat.pkl'))
D_raw = np.load(os.path.join(OUT, 'D_G_cat_temporal_raw.npy'))
code = json.load(open(os.path.join(OUT, 'category_codes.json')))
code_to_label = [None] * len(code)
for k, v in code.items():
    code_to_label[v] = k
D_G_cat = vocab.compute_distance_matrix(D_raw, offset_value=OFFSET)
G28 = D_G_cat.shape[0]

df = fix_clinical_diagnosis(dff.copy())
df = df[~df['participant_id'].isin(incomplete_clinical_administrations.keys())]
df.sort_values('task_number_int', ascending=True, inplace=True)
df = df[df['pathologie'].isin(['HEALTHY', 'RIL', 'TBI'])]
df.drop_duplicates(subset=['trigram'], keep='first', inplace=True)
df = df.reset_index(drop=True)

# frame-level, resampled to L=64 (Goal 1 features are length-invariant histograms/bigrams)
X_cat = np.vstack([upsample_sequence(np.asarray(s), L)
                   for s in df['int_cat_segm_embedding_labels']]).astype(int)
labels = df['pathologie'].values.astype(object)
print('cohort:', df.groupby('pathologie').size().to_dict(), '| X_cat', X_cat.shape)


## Goal 1 — Incremental AUC of bigram ordering over the unigram histogram

Nested CV (outer `RepeatedStratifiedKFold` 10×5, inner `GridSearchCV` on `roc_auc`), paired across
feature sets (`hist` vs `both = hist ⊕ bigram`). Logistic + RF. Reported Δ is `both − hist` with a
paired bootstrap 95% CI over folds. *Caveat:* the bigram block is `G² = 784` features vs n as low as
~60 — read the Δ CI, not the raw `both` AUC.

In [ ]:
folds_io, summ_io = B.evaluate_incremental_ordering(
    X_cat, labels, G28, classifiers=('logreg', 'rf'),
    n_repeats=10, n_folds=5, random_state=42, n_boot=10000,
)
summ_io = summ_io.sort_values(['comparison', 'classifier']).reset_index(drop=True)
folds_io.to_csv(os.path.join(EXP, 'incremental_ordering.csv'), index=False)
summ_io.to_csv(os.path.join(EXP, 'incremental_ordering_summary.csv'), index=False)

show = summ_io.copy()
for c in ['mean_auc_hist', 'mean_auc_both', 'mean_delta', 'delta_ci_low', 'delta_ci_high']:
    show[c] = show[c].round(3)
show['wilcoxon_p'] = show['wilcoxon_p'].round(4)
show['ordering_helps'] = show['delta_ci_low'] > 0
display(show)
print('\n"ordering helps" = incremental-Δ 95% CI excludes 0 (lower bound > 0).')


In [ ]:
# incremental-delta with 95% CI error bars, per comparison x classifier
fig, ax = plt.subplots(figsize=(8, 4))
lab = summ_io['comparison'] + '\n' + summ_io['classifier']
x = np.arange(len(summ_io))
yerr = np.vstack([summ_io['mean_delta'] - summ_io['delta_ci_low'],
                  summ_io['delta_ci_high'] - summ_io['mean_delta']])
ax.bar(x, summ_io['mean_delta'], yerr=yerr, capsize=4,
       color=['C2' if lo > 0 else 'C0' for lo in summ_io['delta_ci_low']])
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=8)
ax.set_ylabel('incremental AUC  (both − hist)')
ax.set_title('Goal 1: does bigram ordering add AUC over the unigram histogram?')
plt.tight_layout()
plt.savefig(os.path.join(EXP, 'incremental_ordering.png'), dpi=130)
plt.show()


## Goal 2 — Pre-registered TBI-vs-RIL follow-up (Edit-Shape DTW + bigram, segment-level)

Restated hypothesis: **Edit-Shape DTW AUC on `RIL_vs_TBI` (segment-level, `step_sequ=1`) beats the
Wasserstein histogram with a paired bootstrap 95% CI on (eshape − hist) whose lower bound > 0.**
Sequences are segment-level (`int_cat_segments_labels`) upsampled to `L_seg = median segment count`
over RIL+TBI — the tractable "full / L=median" form (frame-level L≈5162 is intractable for the O(L²)
Edit-Shape inner rTWE). Builds are deterministic so `n_inits=1` pairs cleanly across methods.

*Runtime note:* `step_sequ=1` at L≈180 is ~1.6 s/distance → the Edit-Shape run is ~25–35 min.

In [ ]:
# RIL+TBI subset, segment-level, upsampled to the median segment count
rt_mask = np.isin(labels, ['RIL', 'TBI'])
seg_rt = [np.asarray(s).astype(int) for s, m in zip(df['int_cat_segments_labels'], rt_mask) if m]
labels_rt = labels[rt_mask]
L_seg = int(np.median([len(s) for s in seg_rt]))
X_seg = np.vstack([upsample_sequence(s, L_seg) for s in seg_rt]).astype(int)
print('RIL+TBI:', {g: int((labels_rt == g).sum()) for g in ['RIL', 'TBI']},
      '| L_seg =', L_seg, '| X_seg', X_seg.shape)

EVAL_RS = 1234
wass = B.default_baseline_methods(D_G_cat, nu=NU, lmbda=LMBDA)['wasserstein']
extra1 = B.extra_experiment_methods(D_G_cat, G28, nu=NU, lmbda=LMBDA, step_sequ=1)
methods_g2 = {
    'wasserstein': wass,
    'transition': extra1['transition'],
    'eshape_dtw': extra1['eshape_dtw'],
}


In [ ]:
# 10 paired splits on the same RIL_vs_TBI subset (this is the long cell: ~25-35 min)
res_g2 = B.evaluate_baselines(X_seg, labels_rt, methods_g2,
                              n_splits=10, n_inits=1, random_state=EVAL_RS)
res_g2.to_csv(os.path.join(EXP, 'tbi_ril_followup.csv'), index=False)
print('per-method mean AUC (RIL_vs_TBI):')
display(res_g2.groupby('method')['auc'].agg(['mean', 'std']).round(3))


In [ ]:
# bootstrap 95% CIs + the pre-registered verdict
COMP = 'RIL_vs_TBI'
ci = {m: B.bootstrap_auc_ci(res_g2, m, COMP, n_boot=10000, random_state=0)
      for m in methods_g2}
ci_tab = pd.DataFrame(
    [{'method': m, 'mean_auc': round(v[0], 3), 'ci_low': round(v[1], 3), 'ci_high': round(v[2], 3)}
     for m, v in ci.items()])
display(ci_tab)

d_es = B.bootstrap_delta_ci(res_g2, 'eshape_dtw', 'wasserstein', COMP, n_boot=10000, random_state=0)
d_tr = B.bootstrap_delta_ci(res_g2, 'transition', 'wasserstein', COMP, n_boot=10000, random_state=0)
verdict_positive = bool(d_es[1] > 0)
result = {
    'comparison': COMP, 'L_seg': L_seg, 'n_splits': 10, 'step_sequ': 1,
    'auc_ci': {m: list(v) for m, v in ci.items()},
    'delta_eshape_minus_hist': {'mean': d_es[0], 'ci_low': d_es[1], 'ci_high': d_es[2]},
    'delta_transition_minus_hist': {'mean': d_tr[0], 'ci_low': d_tr[1], 'ci_high': d_tr[2]},
    'preregistered_hypothesis': 'eshape_dtw > wasserstein on RIL_vs_TBI, delta CI lower bound > 0',
    'verdict_positive': verdict_positive,
}
with open(os.path.join(EXP, 'tbi_ril_followup_ci.json'), 'w') as f:
    json.dump(result, f, indent=2)

print(f'\nΔ eshape−hist : {d_es[0]:+.3f}  95% CI [{d_es[1]:+.3f}, {d_es[2]:+.3f}]')
print(f'Δ transition−hist: {d_tr[0]:+.3f}  95% CI [{d_tr[1]:+.3f}, {d_tr[2]:+.3f}]  (secondary)')
print('\nPRE-REGISTERED VERDICT:',
      'POSITIVE — ordering beats frequency on TBI-vs-RIL (CI excludes histogram).'
      if verdict_positive else
      'NEGATIVE — CI does not exclude the histogram; no ordering advantage. Hand off to Track B.')


## Honest conclusion

The two cells above apply the pre-registered decision rules with no post-hoc tuning:

- **Goal 1 (incremental ordering):** ordering adds discriminative value *only* for a
  `(comparison, classifier)` whose incremental-Δ 95% CI excludes 0 (`ordering_helps == True`).
- **Goal 2 (pre-registered TBI-vs-RIL):** positive *only* if the paired bootstrap 95% CI on
  (Edit-Shape DTW − Wasserstein histogram) has `ci_low > 0`. The bigram transition row is secondary.

If both are negative (the expected outcome given §12), the verdict is: *no temporal/ordering signal
beats symbol frequency on this cohort* — proceed to **Track B** (paper reframing: present TW-TWE+DBA
as a principled categorical-sequence averager with ablation-justified design choices, and qualify the
group-discrimination claim). A narrow positive would be reportable only with the CI that produced it.